# DAQ HAT — Build &amp; Flash

One-stop notebook to build and flash the **BugBuster DAQ HAT** processors:

- **ESP32-P4** (application / acquisition) — flashed directly over USB.
- **ESP32-C6** (display / wireless) — flashed **through the P4** over UART.

Run the **Setup** cell once, then use the P4 and C6 cells as needed.


## Setup


In [ ]:
# =====================================================================
# DAQ HAT flasher — SETUP (run this first, once per session)
# =====================================================================
# Resolves the project paths, ensures pyserial, auto-detects the P4 serial
# port, and defines the helpers used by the two action cells below.
import os, sys, subprocess, shutil
from pathlib import Path

# --- Locate the project tree (this notebook lives in <repo>/Notebooks/) ------
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR if (NB_DIR / "Firmware").is_dir() else NB_DIR.parent
P4_DIR = REPO_ROOT / "Firmware" / "DAQ_HAT" / "ESP32P4"
C6_DIR = REPO_ROOT / "Firmware" / "DAQ_HAT" / "ESP32C6"
FLASH_SCRIPT = C6_DIR / "flash_via_p4.py"
assert P4_DIR.is_dir(),       f"P4 project not found: {P4_DIR}"
assert C6_DIR.is_dir(),       f"C6 project not found: {C6_DIR}"
assert FLASH_SCRIPT.is_file(), f"flash script not found: {FLASH_SCRIPT}"

# --- Ensure pyserial (port detection + the C6 flasher) -----------------------
try:
    from serial.tools import list_ports
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyserial"])
    from serial.tools import list_ports

# --- PlatformIO invocation (robust to PATH differences in the kernel) --------
def _pio_cmd():
    exe = shutil.which("pio") or shutil.which("platformio")
    return [exe] if exe else [sys.executable, "-m", "platformio"]
PIO = _pio_cmd()

# --- Streaming command runner (live output in the notebook) ------------------
def run(cmd, cwd=None):
    cmd = [str(c) for c in cmd]
    print("$ " + " ".join(cmd) + f"\n  (cwd={cwd or os.getcwd()})\n", flush=True)
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"FAILED (exit {proc.returncode}): {' '.join(cmd)}")
    print("\n[ok] done (exit 0)")

# --- Auto-detect the P4 USB-Serial-JTAG debug port ---------------------------
# The P4 exposes an Espressif USB-Serial-JTAG (VID 0x303A / PID 0x1001). That is
# the port used both to upload the P4 firmware (esptool) and to flash the C6
# through the P4 REPL. Note the HS measurement stream is a *different* interface
# (PID 0x4001) and is intentionally NOT matched here.
ESP_VID = 0x303A
USJ_PID = 0x1001

def detect_p4_port(ports):
    # 1) Exact Espressif USB-Serial-JTAG match.
    for p in ports:
        if (p.vid, p.pid) == (ESP_VID, USJ_PID):
            return p.device, "USB-Serial-JTAG (303A:1001)"
    # 2) Any Espressif device advertising a JTAG/serial-debug interface.
    for p in ports:
        if p.vid == ESP_VID and "jtag" in (p.description or "").lower():
            return p.device, f"Espressif JTAG ({p.description})"
    # 3) Fall back to the only available port, if there is exactly one.
    if len(ports) == 1:
        return ports[0].device, "only port present"
    return None, None

print("Available serial ports:")
_ports = list(list_ports.comports())
for p in _ports:
    vid = f"{p.vid:04X}" if p.vid is not None else "----"
    pid = f"{p.pid:04X}" if p.pid is not None else "----"
    print(f"  {p.device:12}  {vid}:{pid}  {p.description}")
if not _ports:
    print("  (none found)")

P4_PORT, _why = detect_p4_port(_ports)
# ---> Override here if auto-detect is wrong:
# P4_PORT = "COM15"

# --- Options -----------------------------------------------------------------
C6_FULL = False   # True = flash bootloader+partitions+app (first-time / recovery)

print("\nREPO_ROOT :", REPO_ROOT)
print("P4_DIR    :", P4_DIR)
print("C6_DIR    :", C6_DIR)
print("PIO       :", " ".join(PIO))
if P4_PORT:
    print(f"P4_PORT   : {P4_PORT}  ({_why})")
else:
    print("P4_PORT   : NOT FOUND — set it manually above (uncomment the override)")
print("C6_FULL   :", C6_FULL)

## Build &amp; flash the ESP32-P4


In [ ]:
# =====================================================================
# Build + flash the ESP32-P4 application processor
# =====================================================================
# `pio run -t upload` builds the firmware and flashes it in one step, then
# the P4 reboots into the new image.
port_args = ["--upload-port", P4_PORT] if P4_PORT else []
run(PIO + ["run", "-e", "esp32p4", "-t", "upload", *port_args], cwd=P4_DIR)


## Build &amp; flash the ESP32-C6 (via the P4)


In [ ]:
# =====================================================================
# Build + flash the ESP32-C6 display MCU, THROUGH the P4
# =====================================================================
# The C6 has no direct USB — it is programmed by the P4 over UART using
# esp-serial-flasher. The P4 must already be running the normal firmware
# (its 'c6flash' REPL command). If unsure, run the P4 cell above first.
assert P4_PORT, "Set P4_PORT in the SETUP cell (the P4 USB-Serial-JTAG COM port)."

# 1) Fresh C6 build (robust pio invocation).
run(PIO + ["run", "-e", "esp32c6"], cwd=C6_DIR)

# 2) Flash the freshly built image via the P4 REPL. We just built above, so
#    pass --no-build (the script can also build itself when run standalone).
full = ["--full"] if C6_FULL else []
run([sys.executable, FLASH_SCRIPT, P4_PORT, "--no-build", *full], cwd=C6_DIR)
